# 01 — Explore Conduit data + auto data dictionary
Loads ALL CSVs in data/ (GeoCSV export + weatherdata (N).csv), merges them.
Output: `outputs/data_dictionary.csv`, EDA report.

In [1]:
import sys, os; sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from config import RAW_CSV_GLOB, OUTPUT_DIR
from loader import load_all_data
pd.set_option('display.width', 160)

df = load_all_data(RAW_CSV_GLOB)
df.head(3)

  loaded 3DFEWSNET_SiteJKUAT_KenyaKiambuJKUATIOTAWS-Conduti@Empathy1.csv: 7,060 rows  2026-08-28 00:00:25+00:00 -> 2026-09-01 23:58:31+00:00
  loaded weatherdata (1).csv: 3,040 rows  2026-07-14 00:00:02+00:00 -> 2026-08-14 23:53:20+00:00
  loaded weatherdata (2).csv: 2,945 rows  2026-06-14 00:01:00+00:00 -> 2026-07-14 23:49:55+00:00
  loaded weatherdata (3).csv: 3,040 rows  2026-05-14 00:00:18+00:00 -> 2026-06-14 23:49:37+00:00
  loaded weatherdata (4).csv: 2,943 rows  2026-04-14 00:00:00+00:00 -> 2026-05-14 23:47:10+00:00
  loaded weatherdata (5).csv: 3,040 rows  2026-03-14 00:00:52+00:00 -> 2026-04-14 23:49:10+00:00
  loaded weatherdata (6).csv: 2,751 rows  2026-02-14 00:00:11+00:00 -> 2026-03-14 23:51:30+00:00
  loaded weatherdata (7).csv: 3,036 rows  2026-01-14 00:01:00+00:00 -> 2026-02-14 23:48:07+00:00
  loaded weatherdata (8).csv: 3,032 rows  2025-12-14 00:00:31+00:00 -> 2026-01-14 23:54:03+00:00
TOTAL: 30,223 rows | 2025-12-14 00:00:31+00:00 -> 2026-09-01 23:58:31+00:00


,rain1,rain2,temp_sht,temp_bmx,temp_mcp,humidity_sht,press_bmx,light_vis,light_ir,uv,wind_spd,wind_dir,wind_gust,wind_gust_dir,heat_idx,wet_bulb_temp,wet_bulb_globe_temp
ts,,,,,,,,,,,,,,,,,
2025-12-14 00:00:31+00:00,0.0,0.0,16.5,16.1,16.3,92.6,849.1,261,254,0.0,0.1,62,0.2,0.2,16.5,15.6,12.3
2025-12-14 00:15:36+00:00,0.0,0.0,16.5,16.1,16.3,92.6,849.2,260,254,0.0,0.1,61,0.2,0.2,16.6,15.6,12.4
2025-12-14 00:30:41+00:00,0.0,0.0,16.5,16.1,16.2,92.6,849.2,260,253,0.0,0.1,48,0.4,0.4,16.6,15.7,12.4


In [ ]:
# Automated EDA report.
try:
    from data_profiling import ProfileReport   # new package: fg-data-profiling
except ImportError:
    try:
        from ydata_profiling import ProfileReport   # old package: ydata-profiling (deprecated)
    except ImportError as e:
        ProfileReport = None
        print('Neither fg-data-profiling nor ydata-profiling is installed, skipping:', e)

if ProfileReport is not None:
    try:
        prof = ProfileReport(df.reset_index(), title='ClimaScope - Conduit EDA', explorative=False)
        prof.to_file('../reports/01_eda_report.html')
        print('EDA report -> reports/01_eda_report.html')
    except Exception as e:
        print('Report generation failed, skipping:', e)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 18/18 [00:00<00:00, 45.41it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

EDA report -> reports/01_eda_report.html


In [ ]:
#Data dictionary (auto: missing%, dtype, stats)
units = {'rain1':'mm/interval','rain2':'mm/interval','temp_sht':'degC','temp_bmx':'degC','temp_mcp':'degC',
         'humidity_sht':'%','press_bmx':'hPa','light_vis':'counts','light_ir':'counts','uv':'index',
         'wind_spd':'m/s','wind_dir':'deg','wind_gust':'m/s','heat_idx':'degC','wet_bulb_temp':'degC',
         'wet_bulb_globe_temp':'degC','health':'#'}
desc = {'rain1':'Tipping-bucket rain gauge 1','rain2':'Tipping-bucket rain gauge 2',
        'temp_sht':'Air temperature (primary)','humidity_sht':'Relative humidity',
        'press_bmx':'Barometric pressure','light_vis':'Ambient light (cloud/sun proxy)',
        'wind_spd':'Wind speed','wind_gust':'Wind gust'}
dd = pd.DataFrame({'unit':[units.get(c,'') for c in df.columns],
                   'dtype':df.dtypes.astype(str).values,
                   'missing_pct':(df.isna().mean()*100).round(2).values,
                   'mean':df.mean(numeric_only=True).round(3).values,
                   'min':df.min(numeric_only=True).values,'max':df.max(numeric_only=True).values,
                   'description':[desc.get(c,'') for c in df.columns]}, index=df.columns)
dd.to_csv(f'{OUTPUT_DIR}/data_dictionary.csv')
print('Data dictionary -> outputs/data_dictionary.csv')
dd.sort_values('missing_pct', ascending=False)

Data dictionary -> outputs/data_dictionary.csv


,unit,dtype,missing_pct,mean,min,max,description
rain1,mm/interval,float64,0.0,0.002,0.0,2.0,Tipping-bucket rain gauge 1
rain2,mm/interval,float64,0.0,0.000,0.0,0.4,Tipping-bucket rain gauge 2
temp_sht,degC,float64,0.0,20.362,9.8,34.2,Air temperature (primary)
temp_bmx,degC,float64,0.0,19.933,9.2,32.9,
temp_mcp,degC,float64,0.0,20.146,9.6,33.0,
humidity_sht,%,float64,0.0,69.913,21.7,98.3,Relative humidity
press_bmx,hPa,float64,0.0,850.672,842.9,856.5,Barometric pressure
light_vis,counts,int64,0.0,406.389,257.0,1365.0,Ambient light (cloud/sun proxy)
light_ir,counts,int64,0.0,1826.629,251.0,11939.0,
uv,index,float64,0.0,0.170,0.0,5.1,


In [ ]:
# Quick visual sanity check (whole record)
cols = [c for c in ['temp_sht','humidity_sht','rain1','light_vis'] if c in df.columns]
df[cols].resample('6h').mean().plot(subplots=True, figsize=(12,8), title='6-hourly resample sanity check')
fig = plt.gcf()
plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/01_quick_timeseries.png', dpi=120)

try:
    from IPython.display import display
    display(fig)
except ImportError:
    pass
plt.close(fig)
daily_rain = df[['rain1','rain2']].clip(lower=0).resample('1D').sum().mean(axis=1)
print(f'Rain days (>=1mm): {int((daily_rain>=1).sum())} / {len(daily_rain)} days')


<Figure size 1200x800 with 4 Axes>

Rain days (>=1mm): 7 / 262 days
